# NeuroGolf 7186.74 | task398: Remove the Index Table

This notebook publishes the public 7186.72 archive plus exactly
one measured improvement: **task118**.

```text
task118 cost: 35,986 -> 35,467
measured gain: +0.014527
expected public score: approximately 7186.74
```

No other private model, private target, or private implementation
hint is included.

## NeuroGolf verification series

1. [One Verified Follow-Up](https://www.kaggle.com/code/yuu111111111/neurogolf-7186-74-one-verified-follow-up)
2. [task398: Remove the Index Table](https://www.kaggle.com/code/yuu111111111/neurogolf-7186-74-task398-remove-the-index-table)
3. [Measuring a Safe Micro-Gain](https://www.kaggle.com/code/yuu111111111/neurogolf-7186-74-measuring-a-safe-micro-gain)
4. [Reproducible Local Scoring Guide](https://www.kaggle.com/code/yuu111111111/neurogolf-reproducible-local-scoring-guide)
5. [A Reproducible Improvement Workflow](https://www.kaggle.com/code/yuu111111111/neurogolf-a-reproducible-improvement-workflow)

Each notebook serves a different purpose. All five publish the same audited
archive so that the educational content never changes the submitted model.

## Score model

NeuroGolf charges initializer elements plus intermediate tensor
bytes. For an exact replacement:

\[
\Delta = \ln(C_\mathrm{old}/C_\mathrm{new}).
\]

Node count and ZIP size are diagnostics, not the objective.

## A complete, reproducible task398 rewrite


            task398 builds a lookup table indirectly. The older route first
            concatenates five source columns with a background column and a
            zero column, then expands a 99-element index initializer through
            `Gather`.

            The index initializer is only this run-length description:

            ```text
            [5] * 24
            + [0, 1, 2, 3, 4]
            + [5] * 20
            + [6] * 50
            ```

            The same table can be constructed directly:

            ```text
            Concat(
                bg_col repeated 24 times,
                top5,
                bg_col repeated 20 times,
                zero_col repeated 50 times,
                axis=2,
            )
            ```

            ONNX permits a variadic `Concat` to reference the same tensor
            repeatedly. This removes the 99-element map and one `Gather`
            without changing the downstream index calculation.

            ```text
            old task398 cost: 5,608
            rewritten cost:   5,439
            measured gain:    +0.030599
            ```

            This task398 explanation is educational history. The common
            submission attached to the current five-notebook series uses the
            latest public archive plus task118 only.

In [ ]:
import numpy as np

top5 = np.arange(50, dtype=np.uint8).reshape(1, 10, 5)
bg_col = np.ones((1, 10, 1), dtype=np.uint8)
zero_col = np.zeros((1, 10, 1), dtype=np.uint8)

top7 = np.concatenate([top5, bg_col, zero_col], axis=2)
table_map = [5] * 24 + list(range(5)) + [5] * 20 + [6] * 50
old_table = np.take(top7, table_map, axis=2)
new_table = np.concatenate(
    [bg_col] * 24 + [top5] + [bg_col] * 20 + [zero_col] * 50,
    axis=2,
)

assert np.array_equal(old_table, new_table)
print("Direct Concat reproduces all", new_table.shape[2], "columns.")

## Validation of the common archive

The shared 400-model archive was checked with:

- canonical task names and ZIP integrity;
- full ONNX model checking;
- the official sanitizer and cost profiler;
- sanitized ONNX Runtime execution;
- available ARC examples;
- changed-task and overfit-pattern audits.

The hidden cell below writes the already validated archive. It
does not rebuild competition models during execution.

In [ ]:
import base64
import hashlib
import zipfile
from pathlib import Path

_PAYLOAD_B64 = (
    "..."
    "eFBLBQYAAAAAkAGQAaBaAACvmgcAAAA="
)

submission = Path("submission.zip")
submission.write_bytes(base64.b64decode(_PAYLOAD_B64))
digest = hashlib.sha256(submission.read_bytes()).hexdigest()
assert digest == "362d63f77b6f1648ccc307e1a744d5189b88240dcf597e6eeceb58790e6c1d22"

expected = [f"task{task:03d}.onnx" for task in range(1, 401)]
with zipfile.ZipFile(submission) as archive:
    assert archive.testzip() is None
    assert archive.namelist() == expected

print("Models:", len(expected))
print("SHA-256:", digest)
print("Expected public score: approximately 7186.74")

## Closing note


            Good graph golf removes representation overhead while preserving
            the rule. Always compare the complete output tensor before
            accepting a cheaper graph.